In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [2]:
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

df = pd.read_csv(url)
df.head().T

,0,1,2,3,4
lead_source,organic_search,social_media,referral,paid_ads,referral
industry,technology,technology,retail,manufacturing,manufacturing
employment_status,employed,employed,employed,student,student
location,europe,south_america,europe,NaN,north_america
annual_income,88160.0,72688.0,44697.0,NaN,24062.0
number_of_courses_viewed,3,3,3,3,4
interaction_count,4,7,4,6,5
lead_score,0.64,0.72,0.58,0.57,0.62
converted,1,1,1,0,1


In [3]:
df.dtypes

,0
lead_source,object
industry,object
employment_status,object
location,object
annual_income,float64
number_of_courses_viewed,int64
interaction_count,int64
lead_score,float64
converted,int64


In [4]:
cat_cols = df.select_dtypes(include='object').columns
num_cols = df.select_dtypes(include='number').columns

df[cat_cols] = df[cat_cols].fillna('NA')
df[num_cols] = df[num_cols].fillna(0.0)

df.isnull().sum()

,0
lead_source,0
industry,0
employment_status,0
location,0
annual_income,0
number_of_courses_viewed,0
interaction_count,0
lead_score,0
converted,0


In [5]:
len(cat_cols), len(num_cols)

(4, 5)

In [6]:
df.industry.mode()

,industry
0,technology


In [7]:
from sklearn.model_selection import train_test_split

In [8]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [9]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [10]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [11]:
from sklearn.metrics import mutual_info_score

In [12]:
numerical = ['interaction_count', 'lead_score', 'number_of_courses_viewed', 'annual_income']

In [13]:
df_full_train.interaction_count.corr(df_full_train.lead_score)

np.float64(0.9166258153290466)

In [14]:
df_full_train.number_of_courses_viewed.corr(df_full_train.lead_score)

np.float64(0.7571778868541085)

In [15]:
df_full_train.number_of_courses_viewed.corr(df_full_train.interaction_count)

np.float64(0.7230400573722874)

In [16]:
df_full_train.annual_income.corr(df_full_train.interaction_count)

np.float64(0.12252817106738928)

In [17]:
def mutual_info_converted_score(series):
  return mutual_info_score(series, df_full_train.converted)

In [18]:
mi = df_full_train[cat_cols].apply(mutual_info_converted_score)
mi.sort_values(ascending=False)

,0
lead_source,0.034761
employment_status,0.020313
industry,0.002898
location,0.001050


In [19]:
df_train[cat_cols].nunique()

,0
lead_source,6
industry,7
employment_status,5
location,6


In [20]:
from sklearn.feature_extraction import DictVectorizer

In [21]:
dv = DictVectorizer(sparse=False)

In [22]:
feature_cols = [c for c in df_full_train.columns if c != 'converted']

In [23]:
train_dicts = df_train[feature_cols].to_dict(orient='records')

In [24]:
X_train = dv.fit_transform(train_dicts)

In [25]:
dv.get_feature_names_out()

array(['annual_income', 'employment_status=NA',
       'employment_status=employed', 'employment_status=self_employed',
       'employment_status=student', 'employment_status=unemployed',
       'industry=NA', 'industry=education', 'industry=finance',
       'industry=healthcare', 'industry=manufacturing', 'industry=retail',
       'industry=technology', 'interaction_count', 'lead_score',
       'lead_source=NA', 'lead_source=events',
       'lead_source=organic_search', 'lead_source=paid_ads',
       'lead_source=referral', 'lead_source=social_media', 'location=NA',
       'location=africa', 'location=asia', 'location=europe',
       'location=north_america', 'location=south_america',
       'number_of_courses_viewed'], dtype=object)

In [26]:
X_train.shape

(3000, 28)

In [27]:
val_dicts = df_val[feature_cols].to_dict(orient='records')

In [28]:
X_val = dv.transform(val_dicts)

In [29]:
from sklearn.linear_model import LogisticRegression

In [30]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

In [31]:
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [32]:
model.intercept_[0]

np.float64(-0.010838978497809288)

In [33]:
model.coef_.round(3)

array([[-0.   , -0.001,  0.007, -0.003, -0.008, -0.006, -0.   , -0.003,
        -0.001, -0.002, -0.002, -0.002, -0.001,  0.137,  0.011,  0.   ,
        -0.009,  0.007, -0.014,  0.009, -0.004,  0.   , -0.002, -0.002,
        -0.001, -0.003, -0.004,  0.054]])

In [34]:
y_pred = model.predict_proba(X_val)[:, 1]

In [35]:
converted_decision = (y_pred >= 0.5)

In [36]:
df_val[converted_decision]

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score
0,NA,technology,employed,asia,80457.0,3,5,0.64
2,paid_ads,technology,student,africa,43944.0,2,4,0.49
3,organic_search,finance,employed,europe,73795.0,2,6,0.68
4,paid_ads,finance,NA,north_america,76619.0,1,3,0.40
5,referral,retail,self_employed,south_america,52392.0,5,9,0.93
...,...,...,...,...,...,...,...,...
994,organic_search,finance,employed,south_america,77662.0,4,6,0.73
995,organic_search,retail,employed,europe,46912.0,3,6,0.66
996,paid_ads,technology,student,north_america,0.0,1,5,0.48
998,events,healthcare,student,north_america,40878.0,1,3,0.39


In [37]:
original_acc = (y_val == converted_decision).mean()
original_acc.round(3)

np.float64(0.645)

**Задание 5**

In [38]:
for feature in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
  train_dicts = df_train[[f for f in feature_cols if f != feature]].to_dict(orient='records')
  val_dicts = df_val[[f for f in feature_cols if f != feature]].to_dict(orient='records')

  X_train = dv.fit_transform(train_dicts)
  X_val = dv.transform(val_dicts)

  model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

  model.fit(X_train, y_train)

  y_pred = model.predict_proba(X_val)[:, 1]

  converted_decision = (y_pred >= 0.5)

  acc = (y_val == converted_decision).mean()
  print(f'{feature}: {original_acc - acc}')

lead_source: 0.0030000000000000027
number_of_courses_viewed: 0.0020000000000000018
interaction_count: 0.04400000000000004


**Задание 6**

In [39]:
dv = DictVectorizer(sparse=False)

train_dicts = df_train[feature_cols].to_dict(orient='records')
val_dicts = df_val[feature_cols].to_dict(orient='records')

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

for c in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]

    converted_decidion = (y_pred >= 0.5)
    acc = (converted_decidion == y_val).mean()
    print(f'C={c}: {acc.round(5)}')

C=1e-06: 0.598
C=1e-05: 0.598
C=0.0001: 0.613
C=0.001: 0.645
